# Patient-Level Bootstrap Confidence Intervals

Resample patients with replacement and include all windows from each sampled
patient, following the rebuttal analysis. Metrics are evaluated on the pooled
windows; scores are not averaged per patient. Stratification classifies a patient
as positive if any of their windows is positive. Validation thresholds stay fixed.


In [ ]:
import json
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from raven.bootstrap import patient_bootstrap_ci, align_legacy_predictions


## Configuration

In [ ]:
# Point to the model directory containing the condition evaluation folders.
out_dir = './output'
output_csv = './eval_data/bootstrap_ci_patient_level.csv'
conditions = ['COPD', 'CHF', 'Dementia', 'Pancreatic Cancer', 'Breast Cancer',
              'Prostate Cancer', 'Heart Attack', 'Knee Osteoarthritis', 'Stroke']
time_horizons = [730, 1825]
approach = 'direct'
n_bootstraps = 1000
bootstrap_type = 'stratified'  # Or 'standard': sample all patients together.
confidence_level = 0.95
random_seed = 42
n_jobs = 1  # Set to the number of CPU cores allocated to this analysis.

# New evaluations save aligned patient identifiers in scores_and_labels.npz.
# For older full-cohort files without identifiers, configure each run explicitly.
# layout='legacy_periodic': original/rebuttal evaluator (native keys, padded
# sampler, periodic gathers). layout='rank_concatenated': optimized evaluator
# (sorted keys, disjoint shards, final rank gather). Do not guess these settings.
legacy_alignment = {
    # ('COPD', 730): dict(data_folder='./eval_data/COPD', layout='legacy_periodic',
    #                    world_size=2, batch_size=8, n_gather=100),
}


## Helper Functions

In [ ]:
def load_results(out_dir, condition_name, time_horizon, approach):
    results_dir = os.path.join(out_dir, f'{condition_name}_evaluation_results_{time_horizon}_{approach}')
    if not os.path.isdir(results_dir):
        return None
    with np.load(os.path.join(results_dir, 'scores_and_labels.npz'), allow_pickle=False) as data:
        scores = data['test_scores'].reshape(-1)
        labels = data['test_labels'].reshape(-1)
        if 'test_patient_ids' in data:
            patient_ids = data['test_patient_ids'].copy()
        else:
            alignment = legacy_alignment.get((condition_name, time_horizon))
            if alignment is None:
                raise ValueError(f'{condition_name}/{time_horizon}: missing patient identifiers. '
                                 'Rerun evaluation or configure validated legacy_alignment above.')
            scores, labels, patient_ids = align_legacy_predictions(scores, labels, **alignment)
    with open(os.path.join(results_dir, 'metrics.json')) as handle:
        metrics = json.load(handle)
    return scores, labels, patient_ids, metrics['optimal_threshold']


## Run Patient-Level Bootstrap


In [ ]:
all_results = []
for condition in tqdm(conditions):
    for horizon in time_horizons:
        loaded = load_results(out_dir, condition, horizon, approach)
        if loaded is None:
            print(f'Skipping {condition}/{horizon}d: results not found')
            continue
        scores, labels, patient_ids, threshold = loaded
        ci, n_patients, n_positive_patients = patient_bootstrap_ci(
            scores, labels, patient_ids, threshold, n_bootstraps=n_bootstraps,
            bootstrap_type=bootstrap_type, confidence_level=confidence_level,
            random_seed=random_seed, n_jobs=n_jobs,
        )
        print(f'{condition}/{horizon}d: {len(labels)} windows, {n_patients} patients')
        for metric_name, stats in ci.items():
            all_results.append(dict(
                condition=condition, time_horizon=horizon, metric=metric_name,
                point_estimate=stats['point_estimate'], ci_lower=stats['lower'],
                ci_upper=stats['upper'], boot_mean=stats['mean'], boot_std=stats['std'],
                n_valid=stats['n_valid'], n_windows=len(labels), n_patients=n_patients,
                n_positive_patients=n_positive_patients, bootstrap_unit='patient',
                bootstrap_type=bootstrap_type, n_bootstraps=n_bootstraps,
                confidence_level=confidence_level, random_seed=random_seed,
            ))


## Results Table

In [ ]:
if not all_results:
    raise ValueError('No evaluation results found; check out_dir and the selected conditions.')
df = pd.DataFrame(all_results)
df['ci_str'] = df.apply(lambda row: f"{row['point_estimate']:.4f} [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]", axis=1)
pivot = df.pivot_table(index=['condition', 'time_horizon'], columns='metric', values='ci_str', aggfunc='first')
pivot = pivot[['auroc', 'auprc', 'f1', 'precision', 'recall']]
pivot


In [ ]:
os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)
df.to_csv(output_csv, index=False)
pivot.to_csv(os.path.splitext(output_csv)[0] + '_formatted.csv')


## Raw DataFrame (for further analysis / export)

In [ ]:
df[['condition', 'time_horizon', 'metric', 'point_estimate', 'ci_lower',
    'ci_upper', 'n_windows', 'n_patients', 'n_positive_patients', 'n_valid']]
